# Broyles EPD — A2 GWP Mapping

Mapping of median A2 (transport) GWP per cubic yard by concrete plant location, overlaid on a greyscale map of the continental US.

### Imports

In [13]:
import os
import pathlib
import pandas as pd
import numpy as np
import plotly.express as px

### Load Data

In [14]:
cwd = pathlib.Path(os.getcwd())
repo_root = cwd.parent if cwd.name == '04_analysis_scripts' else cwd

csv_path = repo_root / '02_processed_data' / 'broyles_epd_data_cleaned.csv'
df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} records")
df[['Plant', 'Plant Location - City', 'Plant Location - State',
    'Plant Location - Zip', 'plant_lat', 'plant_lon', 'A2 GWP_per_CY']].head()

Loaded 44,327 records


,Plant,Plant Location - City,Plant Location - State,Plant Location - Zip,plant_lat,plant_lon,A2 GWP_per_CY
0,Boardman Plant,Boardman,OR,97818,45.8272,-119.7206,4.01
1,Boardman Plant,Boardman,OR,97818,45.8272,-119.7206,7.65
2,Hermiston Plant,Hermiston,OR,97838,45.8450,-119.2849,7.80
3,Hermiston Plant,Hermiston,OR,97838,45.8450,-119.2849,24.01
4,Pendleton Plant,Pendleton,OR,97801,45.6605,-118.7831,21.87


### Aggregate Median A2 GWP by Plant Location

Filter to continental US bounds, drop records missing coordinates or A2 GWP, then collapse to one median value per unique plant lat/lon.

In [15]:
df_map = df.dropna(subset=['plant_lat', 'plant_lon', 'A2 GWP']).copy()
df_map = df_map[
    df_map['plant_lat'].between(24.0, 50.0) &
    df_map['plant_lon'].between(-125.0, -66.0)
].copy()

df_agg = (
    df_map
    .groupby(['plant_lat', 'plant_lon'], as_index=False)
    .agg(
        median_a2_gwp=('A2 GWP', 'median'),
        company=('Company', 'first'),
        city=('Plant Location - City', 'first'),
        state=('Plant Location - State', 'first'),
    )
)

df_agg['hover_text'] = (
    'Concrete Supplier: ' + df_agg['company'].fillna('') + '<br>' +
    'Plant Location: ' + df_agg['city'].fillna('') + ', ' + df_agg['state'].fillna('')
)

print(f"Unique plant locations: {len(df_agg):,}")
print(f"A2 GWP/m³ range: {df_agg['median_a2_gwp'].min():.2f} – {df_agg['median_a2_gwp'].max():.2f} kgCO2e/m³")
df_agg.head()

Unique plant locations: 594
A2 GWP/m³ range: 1.71 – 186.00 kgCO2e/m³


,plant_lat,plant_lon,median_a2_gwp,company,city,state,hover_text
0,25.2846,-80.6246,62.30,CEMEX,Florida City,FL,Concrete Supplier: CEMEX<br>Plant Location: Fl...
1,25.7392,-80.3103,62.40,CEMEX,Miami,FL,Concrete Supplier: CEMEX<br>Plant Location: Mi...
2,25.7864,-80.2042,103.00,CEMEX,Miami,FL,Concrete Supplier: CEMEX<br>Plant Location: Mi...
3,25.7877,-80.4166,61.65,CEMEX,Miami,FL,Concrete Supplier: CEMEX<br>Plant Location: Mi...
4,25.8130,-80.2320,95.60,CEMEX,Miami,FL,Concrete Supplier: CEMEX<br>Plant Location: Mi...


### Approach 1: Spike Map (Scattermap polygons)

Filled triangle spikes drawn as lat/lon polygon paths on top of the same carto-positron tile basemap. Each spike has a fixed base width (±longitude offset) and a height scaled by median A2 GWP. Spikes are grouped into 20 color bins so each `go.Scattermap` trace gets one solid fill color from the YlGnBu colorscale. A separate invisible dot layer handles hover text.

In [16]:
import plotly.graph_objects as go
import plotly.colors as pc

ylgnbu_trimmed = [
    [0.000, '#c7e9b4'],
    [0.143, '#7fcdbb'],
    [0.286, '#41b6c4'],
    [0.429, '#1d91c0'],
    [0.571, '#225ea8'],
    [0.714, '#253494'],
    [1.000, '#081d58'],
]

gwp_min = df_agg['median_a2_gwp'].min()
gwp_max = df_agg['median_a2_gwp'].max()

BASE_HALF_WIDTH = 0.18  # longitude degrees, fixed for all spikes
MAX_HEIGHT = 5.0        # latitude degrees at max GWP
MIN_HEIGHT = 0.2        # minimum visible spike
N_BINS = 20
FILL_ALPHA = 0.75

t = (df_agg['median_a2_gwp'] - gwp_min) / (gwp_max - gwp_min)
heights = MIN_HEIGHT + (MAX_HEIGHT - MIN_HEIGHT) * t
bin_idx = (t * N_BINS).clip(0, N_BINS - 1).astype(int)

colors = pc.sample_colorscale(ylgnbu_trimmed, [i / (N_BINS - 1) for i in range(N_BINS)])

def to_rgba(color, alpha):
    # sample_colorscale returns 'rgb(r, g, b)' strings
    return color.replace('rgb(', 'rgba(').replace(')', f', {alpha})')

fig_spikes = go.Figure()

for b in range(N_BINS):
    sub = df_agg[bin_idx == b]
    h_sub = heights[bin_idx == b]
    if sub.empty:
        continue
    lats, lons = [], []
    for (_, row), h in zip(sub.iterrows(), h_sub):
        lats += [row['plant_lat'], row['plant_lat'] + h, row['plant_lat'], None]
        lons += [row['plant_lon'] - BASE_HALF_WIDTH, row['plant_lon'], row['plant_lon'] + BASE_HALF_WIDTH, None]
    fig_spikes.add_trace(go.Scattermap(
        lat=lats, lon=lons,
        mode='lines', fill='toself',
        fillcolor=to_rgba(colors[b], FILL_ALPHA),
        line=dict(color=colors[b], width=0.3),
        showlegend=False, hoverinfo='skip',
    ))

# Invisible dot layer for hover
fig_spikes.add_trace(go.Scattermap(
    lat=df_agg['plant_lat'],
    lon=df_agg['plant_lon'],
    mode='markers',
    marker=dict(size=6, opacity=0),
    text=df_agg['hover_text'],
    customdata=df_agg[['median_a2_gwp']].values,
    hovertemplate='%{text}<br>Median A2 GWP: %{customdata[0]:.2f} kgCO2e/m³<extra></extra>',
    showlegend=False,
))

# Invisible point to drive the colorbar (size=1 not 0.001 — plotly skips colorbar for sub-pixel markers)
fig_spikes.add_trace(go.Scattermap(
    lat=[37.5], lon=[-96],
    mode='markers',
    marker=dict(
        size=1, opacity=0,
        color=[gwp_min], cmin=gwp_min, cmax=gwp_max,
        colorscale=ylgnbu_trimmed,
        showscale=True,
        colorbar=dict(title='kgCO2e/m³', thickness=15, len=0.6),
    ),
    showlegend=False, hoverinfo='skip',
))

fig_spikes.update_layout(
    title=dict(
        text='Median A2 Transport GWP by Concrete Plant Location (kgCO2e/m³)',
        x=0.5,
        xanchor='center',
        font=dict(family='Arial', size=16, weight='bold'),
    ),
    map=dict(
        style='carto-positron',
        center=dict(lat=39.5, lon=-98.35),
        zoom=3.2,
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    width=1400,
    height=900,
)

### Export Spike Map HTML

In [17]:
html_path = repo_root / 'tests' / 'broyles_a2_gwp_spikes.html'
fig_spikes.write_html(str(html_path), include_plotlyjs='cdn')
print(f"Saved HTML to {html_path}")

Saved HTML to c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes.html


### Export Spike Map PNG

In [18]:
# from kaleido import Kaleido

# png_path = repo_root / 'tests' / 'broyles_a2_gwp_spikes.png'
# async with Kaleido(n=4) as k:
#     await k.write_fig(fig_spikes, path=str(png_path),
#                       opts={'format': 'png', 'scale': 5,
#                             'width': fig_spikes.layout.width, 'height': fig_spikes.layout.height})
# print(f"Saved PNG to {png_path}")

### Approach 2: Metro-Consolidated Spike Map

Same spike style as Approach 1b, but aggregated to one spike per metro area centroid. Records where `Metro/State (within 60 mi)` was a bare state name retain their original plant lat/lon, so they still appear as individual spikes. Records assigned to a named metro area are collapsed to the metro centroid with a single median A2 GWP value.

#### Aggregate Median A2 GWP by Metro Location

In [19]:
df_map_metro = df.dropna(subset=['metro_lat', 'metro_lon', 'A2 GWP']).copy()
df_map_metro = df_map_metro[
    df_map_metro['metro_lat'].between(24.0, 50.0) &
    df_map_metro['metro_lon'].between(-125.0, -66.0)
].copy()

df_agg_metro = (
    df_map_metro
    .groupby(['metro_lat', 'metro_lon'], as_index=False)
    .agg(
        median_a2_gwp=('A2 GWP', 'median'),
        metro_area=('Metro/State (within 60 mi)', 'first'),
        n_mixes=('A2 GWP', 'count'),
    )
)

df_agg_metro['hover_text'] = (
    df_agg_metro['metro_area'].fillna('Unknown') +
    '<br>N mixes: ' + df_agg_metro['n_mixes'].astype(str)
)

print(f"Unique metro locations: {len(df_agg_metro):,}")
print(f"A2 GWP/m³ range: {df_agg_metro['median_a2_gwp'].min():.2f} – {df_agg_metro['median_a2_gwp'].max():.2f} kgCO2e/m³")
df_agg_metro.head()

Unique metro locations: 249
A2 GWP/m³ range: 1.71 – 186.00 kgCO2e/m³


,metro_lat,metro_lon,median_a2_gwp,metro_area,n_mixes,hover_text
0,25.7617,-80.1918,83.65,"Miami-Fort Lauderdale-Pompano Beach, FL",326,"Miami-Fort Lauderdale-Pompano Beach, FL<br>N m..."
1,25.9221,-97.4612,186.00,Texas,11,Texas<br>N mixes: 11
2,26.0143,-81.5856,69.60,Florida,31,Florida<br>N mixes: 31
3,26.1337,-97.6447,105.50,Texas,12,Texas<br>N mixes: 12
4,26.1529,-81.7417,99.10,Florida,22,Florida<br>N mixes: 22


#### Build Metro Spike Map

In [20]:
gwp_min_m = df_agg_metro['median_a2_gwp'].min()
gwp_max_m = df_agg_metro['median_a2_gwp'].max()

t_m = (df_agg_metro['median_a2_gwp'] - gwp_min_m) / (gwp_max_m - gwp_min_m)
heights_m = MIN_HEIGHT + (MAX_HEIGHT - MIN_HEIGHT) * t_m
bin_idx_m = (t_m * N_BINS).clip(0, N_BINS - 1).astype(int)

fig_spikes_metro = go.Figure()

for b in range(N_BINS):
    sub = df_agg_metro[bin_idx_m == b]
    h_sub = heights_m[bin_idx_m == b]
    if sub.empty:
        continue
    lats, lons = [], []
    for (_, row), h in zip(sub.iterrows(), h_sub):
        lats += [row['metro_lat'], row['metro_lat'] + h, row['metro_lat'], None]
        lons += [row['metro_lon'] - BASE_HALF_WIDTH, row['metro_lon'], row['metro_lon'] + BASE_HALF_WIDTH, None]
    fig_spikes_metro.add_trace(go.Scattermap(
        lat=lats, lon=lons,
        mode='lines', fill='toself',
        fillcolor=to_rgba(colors[b], FILL_ALPHA),
        line=dict(color=colors[b], width=0.3),
        showlegend=False, hoverinfo='skip',
    ))

# Invisible dot layer for hover
fig_spikes_metro.add_trace(go.Scattermap(
    lat=df_agg_metro['metro_lat'],
    lon=df_agg_metro['metro_lon'],
    mode='markers',
    marker=dict(size=6, opacity=0),
    text=df_agg_metro['hover_text'],
    customdata=df_agg_metro[['median_a2_gwp']].values,
    hovertemplate='%{text}<br>Median A2 GWP: %{customdata[0]:.2f} kgCO2e/m³<extra></extra>',
    showlegend=False,
))

# Invisible point to drive the colorbar (size=1 not 0.001 — plotly skips colorbar for sub-pixel markers)
fig_spikes_metro.add_trace(go.Scattermap(
    lat=[37.5], lon=[-96],
    mode='markers',
    marker=dict(
        size=1, opacity=0,
        color=[gwp_min_m], cmin=gwp_min_m, cmax=gwp_max_m,
        colorscale=ylgnbu_trimmed,
        showscale=True,
        colorbar=dict(title='kgCO2e/m³', thickness=15, len=0.6),
    ),
    showlegend=False, hoverinfo='skip',
))

fig_spikes_metro.update_layout(
    title=dict(
        text='Median A2 Transport GWP by Metro Area (kgCO2e/m³)',
        x=0.5,
        xanchor='center',
        font=dict(family='Arial', size=16, weight='bold'),
    ),
    map=dict(
        style='carto-positron',
        center=dict(lat=39.5, lon=-98.35),
        zoom=3.2,
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    width=1400,
    height=900,
)

### Export Metro Spike Map HTML

In [21]:
html_path = repo_root / 'tests' / 'broyles_a2_gwp_spikes_metro.html'
fig_spikes_metro.write_html(str(html_path), include_plotlyjs='cdn')
print(f"Saved HTML to {html_path}")

Saved HTML to c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_metro.html


### Export Metro Spike Map PNG

In [22]:
# from kaleido import Kaleido

# png_path = repo_root / 'tests' / 'broyles_a2_gwp_spikes_metro.png'
# async with Kaleido(n=4) as k:
#     await k.write_fig(fig_spikes_metro, path=str(png_path),
#                       opts={'format': 'png', 'scale': 5,
#                             'width': fig_spikes_metro.layout.width, 'height': fig_spikes_metro.layout.height})
# print(f"Saved PNG to {png_path}")

### SCM-Breakdown Spike Maps (Plant-Level)

Four separate spike maps — one per SCM bucket — using the full plant lat/lon list (no metro consolidation). Each map uses the same colorscale and spike geometry as Approach 1, but the GWP scale is normalized independently within each bucket.

In [23]:
def build_scm_spike_map(df_sub, title, filename):
    df_sub_agg = (
        df_sub
        .groupby(['plant_lat', 'plant_lon'], as_index=False)
        .agg(
            median_a2_gwp=('A2 GWP', 'median'),
            company=('Company', 'first'),
            city=('Plant Location - City', 'first'),
            state=('Plant Location - State', 'first'),
        )
    )
    df_sub_agg['hover_text'] = (
        'Concrete Supplier: ' + df_sub_agg['company'].fillna('') + '<br>' +
        'Plant Location: ' + df_sub_agg['city'].fillna('') + ', ' + df_sub_agg['state'].fillna('')
    )

    if df_sub_agg.empty:
        print(f"No data for: {title}")
        return None

    gwp_min_s = df_sub_agg['median_a2_gwp'].min()
    gwp_max_s = df_sub_agg['median_a2_gwp'].max()

    if gwp_max_s > gwp_min_s:
        t_s = (df_sub_agg['median_a2_gwp'] - gwp_min_s) / (gwp_max_s - gwp_min_s)
    else:
        t_s = pd.Series(0.5, index=df_sub_agg.index)

    heights_s = MIN_HEIGHT + (MAX_HEIGHT - MIN_HEIGHT) * t_s
    bin_idx_s = (t_s * N_BINS).clip(0, N_BINS - 1).astype(int)

    fig = go.Figure()

    for b in range(N_BINS):
        sub = df_sub_agg[bin_idx_s == b]
        h_sub = heights_s[bin_idx_s == b]
        if sub.empty:
            continue
        lats, lons = [], []
        for (_, row), h in zip(sub.iterrows(), h_sub):
            lats += [row['plant_lat'], row['plant_lat'] + h, row['plant_lat'], None]
            lons += [row['plant_lon'] - BASE_HALF_WIDTH, row['plant_lon'], row['plant_lon'] + BASE_HALF_WIDTH, None]
        fig.add_trace(go.Scattermap(
            lat=lats, lon=lons,
            mode='lines', fill='toself',
            fillcolor=to_rgba(colors[b], FILL_ALPHA),
            line=dict(color=colors[b], width=0.3),
            showlegend=False, hoverinfo='skip',
        ))

    fig.add_trace(go.Scattermap(
        lat=df_sub_agg['plant_lat'],
        lon=df_sub_agg['plant_lon'],
        mode='markers',
        marker=dict(size=6, opacity=0),
        text=df_sub_agg['hover_text'],
        customdata=df_sub_agg[['median_a2_gwp']].values,
        hovertemplate='%{text}<br>Median A2 GWP: %{customdata[0]:.2f} kgCO2e/m³<extra></extra>',
        showlegend=False,
    ))

    fig.add_trace(go.Scattermap(
        lat=[37.5], lon=[-96],
        mode='markers',
        marker=dict(
            size=1, opacity=0,
            color=[gwp_min_s], cmin=gwp_min_s, cmax=gwp_max_s,
            colorscale=ylgnbu_trimmed,
            showscale=True,
            colorbar=dict(title='kgCO2e/m³', thickness=15, len=0.6),
        ),
        showlegend=False, hoverinfo='skip',
    ))

    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor='center',
            font=dict(family='Arial', size=16, weight='bold'),
        ),
        map=dict(
            style='carto-positron',
            center=dict(lat=39.5, lon=-98.35),
            zoom=3.2,
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        width=1400,
        height=900,
    )

    html_path = repo_root / 'tests' / filename
    fig.write_html(str(html_path), include_plotlyjs='cdn')
    print(f"  {len(df_sub_agg):,} plant locations → {html_path}")
    return fig

In [24]:
scm_buckets = [
    (
        df_map[~df_map['contains_fly_ash'] & ~df_map['contains_slag']],
        'Median A2 Transport GWP — No SCMs (kgCO2e/m³)',
        'broyles_a2_gwp_spikes_no_scm.html',
    ),
    (
        df_map[df_map['contains_fly_ash'] & ~df_map['contains_slag']],
        'Median A2 Transport GWP — Fly Ash Only (kgCO2e/m³)',
        'broyles_a2_gwp_spikes_fly_ash_only.html',
    ),
    (
        df_map[~df_map['contains_fly_ash'] & df_map['contains_slag']],
        'Median A2 Transport GWP — Slag Only (kgCO2e/m³)',
        'broyles_a2_gwp_spikes_slag_only.html',
    ),
    (
        df_map[df_map['contains_fly_ash'] & df_map['contains_slag']],
        'Median A2 Transport GWP — Fly Ash & Slag (kgCO2e/m³)',
        'broyles_a2_gwp_spikes_fly_ash_and_slag.html',
    ),
]

for df_sub, title, filename in scm_buckets:
    print(title)
    build_scm_spike_map(df_sub, title, filename)

Median A2 Transport GWP — No SCMs (kgCO2e/m³)
  537 plant locations → c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_no_scm.html
Median A2 Transport GWP — Fly Ash Only (kgCO2e/m³)
  498 plant locations → c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_fly_ash_only.html
Median A2 Transport GWP — Slag Only (kgCO2e/m³)
  325 plant locations → c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_slag_only.html
Median A2 Transport GWP — Fly Ash & Slag (kgCO2e/m³)
  148 plant locations → c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_fly_ash_and_slag.html


### SCM-Breakdown 2×2 Grid Map

All four SCM buckets in one figure using Plotly `make_subplots`. The colorscale is shared across all four panels (global GWP min/max), so spike height and color are directly comparable between quadrants. One colorbar sits on the right edge.

In [25]:
# Aggregate each SCM bucket; keep all columns for hover
bucket_specs = [
    (df_map[~df_map['contains_fly_ash'] & ~df_map['contains_slag']], 'No SCMs'),
    (df_map[ df_map['contains_fly_ash'] & ~df_map['contains_slag']], 'Fly Ash Only'),
    (df_map[~df_map['contains_fly_ash'] &  df_map['contains_slag']], 'Slag Only'),
    (df_map[ df_map['contains_fly_ash'] &  df_map['contains_slag']], 'Fly Ash & Slag'),
]

bucket_aggs = []
for df_sub, label in bucket_specs:
    agg = (
        df_sub
        .groupby(['plant_lat', 'plant_lon'], as_index=False)
        .agg(
            median_a2_gwp=('A2 GWP', 'median'),
            company=('Company', 'first'),
            city=('Plant Location - City', 'first'),
            state=('Plant Location - State', 'first'),
        )
    )
    agg['hover_text'] = (
        'Concrete Supplier: ' + agg['company'].fillna('') + '<br>' +
        'Plant Location: ' + agg['city'].fillna('') + ', ' + agg['state'].fillna('')
    )
    bucket_aggs.append((agg, label))
    print(f"  {label}: {len(agg):,} plant locations")

# Global range → shared colorscale across all four panels
gwp_min_g = min(a['median_a2_gwp'].min() for a, _ in bucket_aggs)
gwp_max_g = max(a['median_a2_gwp'].max() for a, _ in bucket_aggs)
print(f"\nGlobal A2 GWP range: {gwp_min_g:.2f} – {gwp_max_g:.2f} kgCO2e/m³")

  No SCMs: 537 plant locations
  Fly Ash Only: 498 plant locations
  Slag Only: 325 plant locations
  Fly Ash & Slag: 148 plant locations

Global A2 GWP range: 0.29 – 191.00 kgCO2e/m³


In [26]:
from plotly.subplots import make_subplots

fig_grid = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'map'}, {'type': 'map'}],
           [{'type': 'map'}, {'type': 'map'}]],
    subplot_titles=[label for _, label in bucket_aggs],
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
)

subplot_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (agg, label), (sr, sc) in zip(bucket_aggs, subplot_positions):
    t_g = (agg['median_a2_gwp'] - gwp_min_g) / (gwp_max_g - gwp_min_g)
    heights_g = MIN_HEIGHT + (MAX_HEIGHT - MIN_HEIGHT) * t_g
    bin_idx_g = (t_g * N_BINS).clip(0, N_BINS - 1).astype(int)

    for b in range(N_BINS):
        sub = agg[bin_idx_g == b]
        h_sub = heights_g[bin_idx_g == b]
        if sub.empty:
            continue
        lats, lons = [], []
        for (_, rd), h in zip(sub.iterrows(), h_sub):
            lats += [rd['plant_lat'], rd['plant_lat'] + h, rd['plant_lat'], None]
            lons += [rd['plant_lon'] - BASE_HALF_WIDTH, rd['plant_lon'], rd['plant_lon'] + BASE_HALF_WIDTH, None]
        fig_grid.add_trace(go.Scattermap(
            lat=lats, lon=lons,
            mode='lines', fill='toself',
            fillcolor=to_rgba(colors[b], FILL_ALPHA),
            line=dict(color=colors[b], width=0.3),
            showlegend=False, hoverinfo='skip',
        ), row=sr, col=sc)

    fig_grid.add_trace(go.Scattermap(
        lat=agg['plant_lat'],
        lon=agg['plant_lon'],
        mode='markers',
        marker=dict(size=6, opacity=0),
        text=agg['hover_text'],
        customdata=agg[['median_a2_gwp']].values,
        hovertemplate='%{text}<br>Median A2 GWP: %{customdata[0]:.2f} kgCO2e/m³<extra></extra>',
        showlegend=False,
    ), row=sr, col=sc)

# Single colorbar driven by an invisible point on the right column
fig_grid.add_trace(go.Scattermap(
    lat=[37.5], lon=[-96],
    mode='markers',
    marker=dict(
        size=1, opacity=0,
        color=[gwp_min_g], cmin=gwp_min_g, cmax=gwp_max_g,
        colorscale=ylgnbu_trimmed,
        showscale=True,
        colorbar=dict(
            title=dict(text='kgCO2e/m³', side='right'),
            thickness=16,
            len=0.85,
            x=1.01,
            y=0.5,
        ),
    ),
    showlegend=False, hoverinfo='skip',
), row=1, col=2)

# Apply basemap style and center to all four map axes
for map_key in ['map', 'map2', 'map3', 'map4']:
    fig_grid.layout[map_key].update(dict(
        style='carto-positron',
        center=dict(lat=39.5, lon=-98.35),
        zoom=2.5,
    ))

# Bold up the subplot title annotations
for ann in fig_grid.layout.annotations:
    ann.update(font=dict(family='Arial', size=13, weight='bold'))

fig_grid.update_layout(
    title=dict(
        text='Median A2 Transport GWP by SCM Type — Plant Level (kgCO2e/m³)',
        x=0.5, xanchor='center',
        font=dict(family='Arial', size=16, weight='bold'),
    ),
    margin=dict(l=0, r=110, t=80, b=0),
    width=1400,
    height=900,
)

html_path = repo_root / 'tests' / 'broyles_a2_gwp_spikes_scm_grid.html'
fig_grid.write_html(str(html_path), include_plotlyjs='cdn')
print(f"Saved grid HTML to {html_path}")

Saved grid HTML to c:\Users\jaredf\Dropbox\EDS Course\scm-mapping\tests\broyles_a2_gwp_spikes_scm_grid.html
